In [4]:
import sys
print(sys.executable)

/home/user/miniconda3/envs/athero-vae/bin/python


In [8]:
import pandas as pd

expr = pd.read_csv("/data/athero-vae/data/processed/expression_matrix.csv", index_col=0)
meth = pd.read_csv("/data/athero-vae/data/processed/methylation_matrix.csv", index_col=0)

print(expr.shape)
print(meth.shape)

(47278, 1202)
(485577, 2404)


In [11]:
meth.head()

,100001.Mvalue,100001.detectionPval,100002.Mvalue,100002.detectionPval,100003.Mvalue,100003.detectionPval,100004.Mvalue,100004.detectionPval,100005.Mvalue,100005.detectionPval,...,101260.Mvalue,101260.detectionPval,101261.Mvalue,101261.detectionPval,101262.Mvalue,101262.detectionPval,101263.Mvalue,101263.detectionPval,101264.Mvalue,101264.detectionPval
ID_REF,,,,,,,,,,,,,,,,,,,,,
cg00000029,-0.248280,0.0,-0.020392,0.0,-0.373721,0.0,-0.237185,0.0,0.284723,0.0,...,0.092756,0.0,-0.570741,0.0,-0.390567,0.0,0.162912,0.0,-0.273485,0.0
cg00000108,3.516344,0.0,3.856286,0.0,4.239439,0.0,3.519516,0.0,3.880296,0.0,...,3.873464,0.0,3.284091,0.0,3.561364,0.0,4.042576,0.0,3.319701,0.0
cg00000109,2.752625,0.0,3.267494,0.0,2.992055,0.0,2.926529,0.0,2.868053,0.0,...,2.877356,0.0,2.353765,0.0,3.144560,0.0,2.777298,0.0,3.468839,0.0
cg00000165,-2.030849,0.0,-1.462111,0.0,-1.464236,0.0,-1.569228,0.0,-1.719939,0.0,...,-1.019750,0.0,-1.782947,0.0,-1.637511,0.0,-2.138793,0.0,-1.970228,0.0
cg00000236,1.069549,0.0,1.146412,0.0,1.340683,0.0,0.962154,0.0,1.104444,0.0,...,1.057207,0.0,1.667247,0.0,1.240644,0.0,0.973126,0.0,1.080876,0.0


In [10]:
expr.head()

,GSM1352002,GSM1352003,GSM1352004,GSM1352005,GSM1352006,GSM1352007,GSM1352008,GSM1352009,GSM1352010,GSM1352011,...,GSM1353194,GSM1353195,GSM1353196,GSM1353197,GSM1353198,GSM1353199,GSM1353200,GSM1353201,GSM1353202,GSM1353203
ID_REF,,,,,,,,,,,,,,,,,,,,,
ILMN_1762337,5.136625,5.089840,5.268244,5.247811,4.984297,5.099019,5.401281,5.497121,5.290908,5.743592,...,5.189426,5.095456,5.250572,5.396671,5.116644,5.357195,5.142840,4.967980,5.166105,5.199865
ILMN_2055271,5.878106,6.867448,6.085611,6.116644,5.960724,7.124572,6.645861,5.981317,6.025220,6.999142,...,5.858105,5.529138,5.762097,6.076816,6.001335,7.164932,5.489062,6.491508,6.892653,5.150740
ILMN_1736007,5.184706,5.258854,4.954293,5.115758,5.122904,5.210427,5.144561,4.999533,4.981587,5.014846,...,4.978505,5.151428,5.232020,5.116558,5.545990,5.135728,5.016644,5.230082,4.971681,5.303597
ILMN_2383229,5.415968,4.915011,5.042886,5.190248,4.900996,5.209508,4.948399,5.052756,5.223667,5.167986,...,5.208470,5.018134,5.015709,5.167865,4.985861,5.104405,4.962391,5.180023,5.113188,4.993768
ILMN_1806310,5.079431,4.985693,5.116782,5.226024,5.183386,4.895404,5.201298,5.025883,5.049895,4.962199,...,5.445138,4.973678,5.043262,5.104420,5.037863,4.983197,5.067597,5.337265,4.987283,5.082369


In [12]:
meth = meth.filter(regex="Mvalue")

In [13]:
print(meth.shape)

(485577, 1202)


In [14]:
meth.columns = meth.columns.str.replace(".Mvalue", "", regex=False)

In [15]:
print(meth.columns[:10])
print(expr.columns[:10])

Index(['100001', '100002', '100003', '100004', '100005', '100006', '100007',
       '100008', '100009', '100010'],
      dtype='object')
Index(['GSM1352002', 'GSM1352003', 'GSM1352004', 'GSM1352005', 'GSM1352006',
       'GSM1352007', 'GSM1352008', 'GSM1352009', 'GSM1352010', 'GSM1352011'],
      dtype='object')


In [16]:
print(expr.columns[:10])
print(meth.columns[:10])

Index(['GSM1352002', 'GSM1352003', 'GSM1352004', 'GSM1352005', 'GSM1352006',
       'GSM1352007', 'GSM1352008', 'GSM1352009', 'GSM1352010', 'GSM1352011'],
      dtype='object')
Index(['100001', '100002', '100003', '100004', '100005', '100006', '100007',
       '100008', '100009', '100010'],
      dtype='object')


In [17]:
import GEOparse

gse = GEOparse.get_GEO("GSE56045", destdir="data/raw")

mapping = {}

for gsm_id, gsm in gse.gsms.items():
    
    title = gsm.metadata['title'][0]
    
    subject_id = title.split("_")[0]   # "100001_peripheral_CD14"
    
    mapping[gsm_id] = subject_id

16-Mar-2026 13:05:32 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE56nnn/GSE56045/soft/GSE56045_family.soft.gz to data/raw/GSE56045_family.soft.gz
100%|██████████| 791M/791M [01:00<00:00, 13.8MB/s]    
16-Mar-2026 13:06:34 DEBUG downloader - Size validation passed
16-Mar-2026 13:06:34 DEBUG downloader - Moving /tmp/tmp_521r_4d to /data/athero-vae/notebooks/data/raw/GSE56045_family.soft.gz
16-Mar-2026 13:06:34 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE56nnn/GSE56045/soft/GSE56045_family.soft.gz
16-Mar-2026 13:06:35 INFO GEOparse - Parsing data/raw/GSE56045_family.soft.gz: 
16-Mar-2026 13:06:35 DEBUG GEOparse - DATABASE: GeoMiame
16-Mar-2026 13:06:35 DEBUG GEOparse - SERIES: GSE56045
16-Mar-2026 13:06:35 DEBUG GEOparse - PLATFORM: GPL10558
16-Mar-2026 13:06:35 DEBUG GEOparse - SAMPLE: GSM1352002
16-Mar-2026 13:06:35 DEBUG GEOparse - SAMPLE: GSM1352003
16-Mar-2026 13:06:35 DEBUG GEOparse - SAMPLE: GSM1352004
16-Mar-2026 13:0

In [18]:
expr.columns = expr.columns.map(mapping)

In [19]:
print(expr.columns[:10])
print(meth.columns[:10])

Index(['100001', '100002', '100003', '100004', '100005', '100006', '100007',
       '100008', '100009', '100010'],
      dtype='object')
Index(['100001', '100002', '100003', '100004', '100005', '100006', '100007',
       '100008', '100009', '100010'],
      dtype='object')


In [20]:
common = expr.columns.intersection(meth.columns)

expr = expr[common]
meth = meth[common]

print(expr.shape)
print(meth.shape)

(47278, 1202)
(485577, 1202)


In [21]:
expr = expr.T

In [22]:
meth = meth.T

In [ ]:
#feature reduction based on variance
expr = expr.loc[:, expr.var().nlargest(5000).index]
meth = meth.loc[:, meth.var().nlargest(20000).index]

In [24]:
print(meth.shape)

(1202, 20000)


In [25]:
print(expr.shape)

(1202, 5000)


In [29]:
import numpy as np

np.save("../data/processed/expression.npy", expr.values)
np.save("../data/processed/methylation.npy", meth.values)

In [33]:
from sklearn.preprocessing import StandardScaler

expr_scaler = StandardScaler()
meth_scaler = StandardScaler()

expr_scaled = expr_scaler.fit_transform(expr)
meth_scaled = meth_scaler.fit_transform(meth)

In [34]:
import pandas as pd

expr_scaled = pd.DataFrame(expr_scaled, index=expr.index, columns=expr.columns)
meth_scaled = pd.DataFrame(meth_scaled, index=meth.index, columns=meth.columns)

In [36]:
import numpy as np

np.save("../data/processed/expression_training.npy", expr_scaled.values)
np.save("../data/processed/methylation_training.npy", meth_scaled.values)

In [37]:
expr.columns.to_series().to_csv("../data/processed/expression_features.csv")
meth.columns.to_series().to_csv("../data/processed/methylation_features.csv")

expr.index.to_series().to_csv("../data/processed/sample_ids.csv")

In [38]:
print(expr_scaled.shape)
print(meth_scaled.shape)

(1202, 5000)
(1202, 20000)
